In [ ]:
%%bash
ls -alRh /kaggle/input/gustavosta-stable-diffusion-prompts-sd2-v2/train_images | head -n10

In [ ]:
%%bash
head -n10 /kaggle/input/gustavosta-stable-diffusion-prompts-sd2-v2/train.csv

In [ ]:
import os

if any('tpu' in var.lower() for var in os.environ):
    print('TPU is available in the Kaggle environment.')
else:
    print('TPU is not available in the Kaggle environment.')


TPU_AVAILABLE = any('tpu' in var.lower() for var in os.environ)

display(TPU_AVAILABLE)

In [ ]:
import torch

In [ ]:
with open('accelerate_config.yaml', 'w') as f:
    f.write('''
    # Configuration file for HuggingFace Accelerate using double GPU strategy

    # Device placement strategy for data parallelism
    placement_strategy: "ddp"

    # Number of GPUs to use
    nprocs: 2

    # Whether to use gradient accumulation
    gradient_accumulation_steps: 1

    # Gradient accumulation on the first device
    gradient_accumulation_device: 0

    # The maximum number of bytes that can be sent at once for each parameter update
    max_bytes_per_parameter: 1048576

    # Whether to use pipeline parallelism
    pipeline: false

    # Whether to use gradient checkpointing
    gradient_checkpointing: false

    # Whether to use CPU Offloading
    cpu_offload: false

    # Whether to use memory offloading
    memory_offload: false

    # Parameters for automatic mixed precision training
    mixed_precision: true
    loss_scale: "dynamic"
    ''')


In [ ]:
import yaml
with open('accelerate_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

In [ ]:
def tpu_aware_device_setting():
    if TPU_AVAILABLE:
        import torch_xla.core.xla_model as xm
        device = xm.xla_device()
    else:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
from accelerate import Accelerator
from torch.utils.data import DataLoader, DistributedSampler

accelerator = Accelerator()
device = accelerator.device

In [ ]:
from IPython.display import clear_output, display
if TPU_AVAILABLE:
    !pip3 install sentence-transformers
else:
    !pip3 install --upgrade sentence-transformers transformers diffusers
clear_output()

In [ ]:
from PIL import Image
import torch
import torch.nn as nn
from transformers import CLIPProcessor, CLIPVisionModel
import os
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import ToTensor
from pathlib import Path


model_path = "/kaggle/input/laion-clip-vit-h-14-laion2b-s32b-b79k/CLIP-ViT-H-14-laion2B-s32B-b79K/"
hidden_dim = 1280

class ImageDataset(Dataset):
    def __init__(self, image_paths):
        self.image_paths = image_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        img = Image.open(image_path)
        return ToTensor()(img), os.path.splitext(os.path.basename(image_path))[0]

import pandas as pd
from sentence_transformers import SentenceTransformer

# Load the CSV file
train_csv_path = "/kaggle/input/gustavosta-stable-diffusion-prompts-sd2-v2/train.csv"
train_data = pd.read_csv(train_csv_path)

# Load the SentenceTransformer model
emb_model = SentenceTransformer("all-MiniLM-L6-v2").to(device)

print('Sentence Transformer loaded on', device)

# Extract the prompts and their corresponding image paths
prompts = train_data["Prompt"].tolist()
image_paths = train_data["image_path"].tolist()

# Generate prompt embeddings
prompt_embeddings = emb_model.encode(prompts)

# Create a dictionary to maintain the correspondence between prompt embeddings and image paths
prompt_image_dict = dict(enumerate(zip(prompt_embeddings, image_paths)))

In [ ]:
del emb_model

import gc
gc.collect()

import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
class CLIPImageEmbedding(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.model = CLIPVisionModel.from_pretrained(model_path)
        self.processor = CLIPProcessor.from_pretrained(model_path)
        self.linear = nn.Linear(hidden_dim, 384)

    def forward(self, images):
        inputs = self.processor(images=images, return_tensors="pt", padding=True).to(device)
        outputs = self.model(**inputs)
        embeddings = self.linear(outputs["pooler_output"])
        return embeddings

In [ ]:
class CosineSimilarityLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x, y):
        cos_sim = cosine_similarity(x, y)
        loss = 1 - cos_sim.mean()
        return loss

In [ ]:
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
from torch.nn.functional import cosine_similarity
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import torchvision

class CustomImageDataset(Dataset):
    def __init__(self, image_paths, prompt_embeddings, transforms=None):
        self.image_paths = image_paths
        self.prompt_embeddings = prompt_embeddings
        self.transforms = transforms

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert("RGB")

        if self.transforms:
            image = self.transforms(image)
        else:
            image = torchvision.transforms.ToTensor()(image)

        prompt_embedding = self.prompt_embeddings[idx]
        return image, prompt_embedding

In [ ]:
%%writefile basictrain.py

def cosine_similarity_loss(pred, target):
    cos = nn.CosineSimilarity(dim=1)
    output = 1 - cos(pred, target).mean()     # make the loss in [0, 2], which looks better
    return output

import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Create datasets and dataloaders
train_dataset = CustomImageDataset(train_image_paths, train_prompt_embeddings)
val_dataset = CustomImageDataset(val_image_paths, val_prompt_embeddings)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=1)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=1)

# Initialize the CLIP image embedding model and move it to the device
clip_image_embedding = CLIPImageEmbedding(model_path).to(device)

# Define optimizer and loss function
optimizer = optim.AdamW(clip_image_embedding.parameters(), lr=learning_rate)
criterion = cosine_similarity_loss

# TensorBoard summary writer
writer = SummaryWriter()

# Training loop
clip_image_embedding.train()
for epoch in range(epochs):
    running_loss = 0.0
    for batch_idx, (images, batch_prompt_embeddings) in enumerate(tqdm(train_dataloader)):
        images = images.to(device)
        batch_prompt_embeddings = batch_prompt_embeddings.to(device)
        optimizer.zero_grad()

        # Generate image embeddings
        image_embeddings = clip_image_embedding(images)

        # Calculate loss using cosine similarity
        loss = criterion(image_embeddings, batch_prompt_embeddings)

        # Backward pass
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Log progress
        if batch_idx % log_interval == 0:
            print(f"Train Epoch: {epoch} [{batch_idx * len(images)}/{len(train_dataloader.dataset)}] Loss: {loss.item()}")

    # Save model checkpoint
    torch.save(clip_image_embedding.state_dict(), f"clip_image_embedding_epoch{epoch}.pt")

    # Log average training loss for the epoch
    writer.add_scalar("Loss/train", running_loss / len(train_dataloader), epoch)

    # Evaluation
    clip_image_embedding.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, batch_prompt_embeddings in val_dataloader:
            images = images.to(device)
            batch_prompt_embeddings = batch_prompt_embeddings.to(device)

            # Generate image embeddings
            image_embeddings = clip_image_embedding(images)

            # Calculate loss using cosine similarity
            loss = criterion(image_embeddings, batch_prompt_embeddings)
            val_loss += loss.item()

    # Log average validation loss for the epoch
    writer.add_scalar("Loss/validation", val_loss / len(val_dataloader), epoch)

# Close TensorBoard writer
writer.close()


In [ ]:
# ...

batch_size = 64
epochs = 2
learning_rate = 1e-3
log_interval = 16

# Split data into train and validation sets
train_image_paths, val_image_paths, train_prompt_embeddings, val_prompt_embeddings = train_test_split(
    image_paths, prompt_embeddings, test_size=0.2, random_state=42
)

# # Create datasets and dataloaders
# train_dataset = CustomImageDataset(train_image_paths, train_prompt_embeddings)
# val_dataset = CustomImageDataset(val_image_paths, val_prompt_embeddings)

# train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=1)
# val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=1)

In [ ]:
clip_image_embedding = CLIPImageEmbedding(model_path)
clip_image_embedding.model.requires_grad_(False)
assert all(
    param.requires_grad for param in clip_image_embedding.model.parameters()
) == False

clear_output()

In [ ]:
clip_image_embedding.to(device)

def cosine_similarity_loss(pred, target):
    cos = nn.CosineSimilarity(dim=1)
    output = 1 - cos(pred, target).mean()     # make the loss in [0, 2], which looks better
    return output

import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Create datasets and dataloaders
train_dataset = CustomImageDataset(train_image_paths, train_prompt_embeddings)
val_dataset = CustomImageDataset(val_image_paths, val_prompt_embeddings)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Initialize the CLIP image embedding model and move it to the device


clip_image_embedding.to(device)

# Define optimizer and loss function
optimizer = optim.AdamW(clip_image_embedding.parameters(), lr=learning_rate)
criterion = cosine_similarity_loss

# Prepare model, optimizer and criterion for distributed training
clip_image_embedding, optimizer, train_dataloader, val_dataloader = accelerator.prepare(clip_image_embedding, optimizer, train_dataloader, val_dataloader)

# TensorBoard summary writer
writer = SummaryWriter()

# Training loop
clip_image_embedding.train()
for epoch in range(epochs):
    running_loss = 0.0
    for batch_idx, (images, batch_prompt_embeddings) in enumerate(tqdm(train_dataloader)):
        images = images.to(device)
        batch_prompt_embeddings = batch_prompt_embeddings.to(device)
        optimizer.zero_grad()

        # Generate image embeddings
        image_embeddings = clip_image_embedding(images)

        # Calculate loss using cosine similarity
        loss = criterion(image_embeddings, batch_prompt_embeddings)

        # Backward pass
        accelerator.backward(loss)
        optimizer.step()

        running_loss += accelerator.gather(loss).item()

        # Log progress
        if batch_idx % log_interval == 0:
            print(f"Train Epoch: {epoch} [{batch_idx * len(images)}/{len(train_dataloader.dataset)}] Loss: {loss.item()}")
            # Save model checkpoint
            torch.save(clip_image_embedding.state_dict(), f"clip_image_embedding_epoch{epoch}.pt")

    # Log average training loss for the epoch
    writer.add_scalar("Loss/train", running_loss / len(train_dataloader), epoch)

    # Evaluation
    clip_image_embedding.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, batch_prompt_embeddings in val_dataloader:
            images = images.to(device)
            batch_prompt_embeddings = batch_prompt_embeddings.to(device)

            # Generate image embeddings
            image_embeddings = clip_image_embedding(images)

            # Calculate loss using cosine similarity
            loss = criterion(image_embeddings, batch_prompt_embeddings)
            val_loss += accelerator.gather(loss).item()

    # Log average validation loss for the epoch
    writer.add_scalar("Loss/validation", val_loss / len(val_dataloader), epoch)

# Close TensorBoard writer
writer.close()
